# Environment Preparation

In [ ]:
import os, sys, copy, glob
import time, random
import numpy as np
import pickle
import logging
import datetime
import json
import gc

import torch
import torch.utils.data as data
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as transforms

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from torchinfo import summary

import scipy
from scipy.stats import entropy
from scipy.spatial.distance import euclidean, cosine
import sklearn
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN

import argparse

In [ ]:
def args_parser():
    parser = argparse.ArgumentParser()
    # Model
    parser.add_argument("--dataset", help="dataset", default='digits', type=str,
                        choices=['digits', 'office', 'domain', 'cifar10', 'cifar100'])
    parser.add_argument("--model", help="training model", default="resnet18", type=str,
                        choices=['cnn','resnet18', 'resnet34', 'resnet50', 'mobilenetv2'])
    parser.add_argument("--lr", help="learning rate", default=5e-4, type=float)
    parser.add_argument("--momentum", help="SGD momentum", default=0.9, type=float)
    parser.add_argument("--wd", help="weight decay", default=1e-5, type=float)
    parser.add_argument("--batch_size", help="batch size", default=64, type=int)
    parser.add_argument('--device', help="cpu, cuda", default="cuda", type=str)
    parser.add_argument("--gpu", help="index of gpu", default=0, type=int)

    # FL
    parser.add_argument("--aggregation", help="aggregation rule", default='fedavg', type=str,
                        choices=['fedavg', 'krum', 'flame'])
    parser.add_argument("--nrounds", help="# global rounds", default=100, type=int)
    parser.add_argument("--epochs", help="# local epochs", default=5, type=int)
    parser.add_argument("--nclients", help="# clients", default=20, type=int)
    parser.add_argument("--fraction", help="fraction of clients", default=1.0, type=float)
    parser.add_argument("--bias", help="degree of label non-iidness", default=1, type=float)
    parser.add_argument('--init_seed', type=int, default=0, help="Random seed")
    parser.add_argument('--partition', type=str, default='noniid', help='the data partitioning strategy, iid or noniid')

    parser.add_argument('--auto_aug', action='store_true', help='whether to apply auto augmentation')
    parser.add_argument('--aug_mult', help="replicate each client's assigned sample indices this many times "
                        "before building the train Dataset, so random augmentations (crop/flip/autoaug) are "
                        "applied to independently-sampled copies each epoch, inflating the effective per-round "
                        "dataset size without adding new raw images", default=10, type=int)

    parser.add_argument('--krum_m', help="number of clients to select for Krum aggregation", default=1, type=int)

    # Adversarial
    parser.add_argument("--adv_type", help="adv type", default='None', type=str,
                        choices=['None', 'CDLS'])
    parser.add_argument("--nbyz", help="# byzantines / # adversarial clients", default=4, type=int)
    parser.add_argument("--feature", help="feature extraction", default='raw', type=str,
                        choices=['raw','tsne','proto'])
    parser.add_argument("--bd_target_label", help="original label targeted by the CDLS backdoor", default=0, type=int)
    parser.add_argument("--bd_partition", help="fraction of a client's target_label samples to replace with the nearest cross-domain donor sample", default=0.5, type=float)
    parser.add_argument("--bd_adv_clients", help="explicit client ids running the CDLS backdoor; defaults to the first --nbyz clients when not set", default=None, type=int, nargs='+')
    parser.add_argument("--bd_domain", help="digits sub-dataset the clients are assigned to", default='mnist', type=str,
                        choices=['mnist', 'mnist_m', 'svhn', 'syn', 'usps'])
    parser.add_argument("--bd_donor_domains", help="digits sub-datasets donor replacement samples are drawn from; defaults to all domains other than --bd_domain", default=None, type=str, nargs='+')
    parser.add_argument("--bd_donor_pool_size", help="max donor samples per domain kept for the nearest-neighbor search", default=1000, type=int)
    parser.add_argument("--bd_max_search", help="max donor pool entries scanned per victim sample when finding the nearest match", default=500, type=int)
    
    # Logging
    parser.add_argument("--data_dir", type=str, required=False, default="/scratch/jmh8504/data/", 
                        choices=['/scratch/jmh8504/data/', '/export/home/jmh8504/data/'],)

    parser.add_argument('--logdir', type=str, required=False, default="/scratch/jmh8504/FL/flbackdoor/logs/",
                        choices=['/scratch/jmh8504/FL/flbackdoor/logs/', '/export/home/jmh8504/FL/flbackdoor/logs/'],)
                        
    parser.add_argument('--log_file_name', type=str, default=None, help='The log file name')
    parser.add_argument('--ckptdir', type=str, required=False, default="/scratch/jmh8504/FL/flbackdoor/saved_models/",
                        choices=['/scratch/jmh8504/FL/flbackdoor/saved_models/', '/export/home/jmh8504/FL/flbackdoor/saved_models/'],)
    
    parser.add_argument('--print_interval', type=int, default=10,
                        help='how many comm round to print results on screen')
    parser.add_argument('--save_interval', type=int, default=10,

                        help='how many rounds do we save the checkpoint one time') 

    args, unknown = parser.parse_known_args() 

    return args

In [ ]:
args = args_parser()
args

# Utils

In [ ]:
def compute_accuracy(model, dataloader):
    was_training = False
    if model.training:
        model.eval()
        was_training = True

    correct, total = 0, 0
    criterion = nn.CrossEntropyLoss()
    loss_collector = []
    with torch.no_grad():
        for batch_idx, (x, target) in enumerate(dataloader):
            x, target = x.cuda(), target.to(dtype=torch.int64).cuda()
            out = model(x)

            loss = criterion(out, target)
            _, pred_label = torch.max(out.data, 1)
            loss_collector.append(loss.item())
            total += x.data.size()[0]
            correct += (pred_label == target.data).sum().item()

        avg_loss = sum(loss_collector) / len(loss_collector)

    if was_training:
        model.train()

    return correct / float(total), avg_loss

In [ ]:
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    random.seed(seed)

In [ ]:
def mkdirs(dirpath):
    try:
        os.makedirs(dirpath)
    except Exception as _:
        pass

# Data

In [ ]:
from shutil import move
from os.path import join
from os import listdir, rmdir
from torchvision.datasets import ImageFolder, DatasetFolder

from data_aug_utils import AutoAugment

from torchvision.datasets import CIFAR10, CIFAR100, DatasetFolder

In [ ]:
class TwoCropTransform:
    """Create two crops of the same image"""
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return [self.transform(x), self.transform(x)]

## Preprocessing

In [ ]:
# create and save to pkl files for digits-5 dataset
# e.g., save_pkl_digit(os.path.join(data_dir,'digits5'), 'svhn', train=False)
def save_pkl_digit(data_dir, filename, train=True):
    images = []
    if train:
        images += glob.glob(os.path.join(data_dir, '{}/train_images'.format(filename), '*.jpg'))
        file_path = './{}_train.pkl'.format(filename)
    else:
        images += glob.glob(os.path.join(data_dir, '{}/test_images'.format(filename), '*.jpg'))
        file_path = './{}_test.pkl'.format(filename)

    data_path = []
    data_label = []

    for img in images:
        tmp = img.split('/')[3:]

        label = tmp[-1][tmp[-1].find('_') + 1]  # the next number next to '_' is the label
        path = '/'.join(tmp)

        data_path.append(path)
        data_label.append(label)

    with open(file_path, 'wb') as file:
        # Pickle the two lists and write them to the file
        pickle.dump((data_path, data_label), file)

    return tmp

In [ ]:
# create and save to pkl files for DomainNet dataset
# e.g., save_pkl_domain(os.path.join(base_dir,'DomainNet'), real, train=False)
def save_pkl_domain(data_dir, filename, train=True):
  images = []
  labels = []
  train_images = []
  test_images = []
  train_labels = []
  test_labels = []

  #label_dict = {'bird': 0, 'feather': 1, 'headphones': 2, 'ice_cream': 3, 'teapot': 4, 'tiger': 5, 'whale': 6,
                #'windmill': 7, 'wine_glass': 8, 'zebra': 9}

  label_dict = {'stop_sign': 0, 'streetlight': 1, 'traffic_light': 2, 'airplane': 3, 'truck': 4, 'bird': 5, 'school_bus': 6,
                'rain': 7, 'tree': 8, 'feather': 9}

  #label_dict = {'stop_sign': 0, 'streetlight': 1, 'traffic_light': 2, 'airplane': 3, 'truck': 4}

  for keys, values in label_dict.items():
    data_path = os.path.join(data_dir, '{}/{}'.format(filename, keys))
    images += glob.glob(os.path.join(data_path, '*.jpg'))
    images += glob.glob(os.path.join(data_path, '*.png'))

    train_images += images[:50]
    test_images += images[50:80]

    tmp_train = [keys] * 50
    tmp_test = [keys] * 30

    train_labels += tmp_train
    test_labels += tmp_test

    images = []

  data_path = []

  if train:
    file_path = './{}_train_n.pkl'.format(filename)
    for img in train_images:
      tmp = img.split('/')[2:]
      path = '/'.join(tmp)
      data_path.append(path)

    pair = list(zip(data_path, train_labels))
    random.shuffle(pair)
    data_path, train_labels = zip(*pair)
  else:
    file_path = './{}_test_n.pkl'.format(filename)
    for img in test_images:
      tmp = img.split('/')[2:]
      path = '/'.join(tmp)
      data_path.append(path)

    pair = list(zip(data_path, test_labels))
    random.shuffle(pair)
    data_path, test_labels = zip(*pair)

  with open(file_path, 'wb') as file:
    # Pickle the two lists and write them to the file
    if train:
      pickle.dump((data_path, train_labels), file)
    else:
      pickle.dump((data_path, test_labels), file)

## Dataset Classes

In [ ]:
class DigitsDataset(Dataset):
    
    def __init__(self, base_path, subset_name, dataidxs=None, train=True, transform=None):
        if train:
            self.paths, self.text_labels = np.load(base_path+'digits5/'+'{}_train.pkl'.format(subset_name), allow_pickle=True)
        else:
            self.paths, self.text_labels = np.load(base_path+'digits5/'+'{}_test.pkl'.format(subset_name), allow_pickle=True)

        label_dict={'0':0, '1':1, '2':2, '3':3, '4':4, '5':5, '6':6, '7':7, '8':8, '9':9}

        self.dataidxs = dataidxs

        self.labels = [label_dict[text] for text in self.text_labels] # transfer str to num
        self.transform = transform
        self.base_path = base_path if base_path is not None else '../data'

        if self.dataidxs is not None:
            self.paths = [self.paths[i] for i in self.dataidxs]
            self.labels = [self.labels[i] for i in self.dataidxs]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.base_path, 'digits5/', self.paths[idx])
        label = self.labels[idx]
        image = Image.open(img_path)

        if len(image.split()) != 3:
            image = transforms.Grayscale(num_output_channels=3)(image)

        image = np.array(image)

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
class DomainNetDataset(Dataset):
    
    def __init__(self, base_path, subset_name, dataidxs=None, train=True, transform=None):
        if train:
            self.paths, self.text_labels = np.load(os.path.join(base_path, 'DomainNet', 'pkls', '{}_train.pkl'.format(subset_name)), allow_pickle=True)
        else:
            self.paths, self.text_labels = np.load(os.path.join(base_path, 'DomainNet', 'pkls', '{}_test.pkl'.format(subset_name)), allow_pickle=True)

        label_dict = {'bird': 0, 'feather': 1, 'headphones': 2, 'ice_cream': 3, 'teapot': 4, 'tiger': 5, 'whale': 6,
                    'windmill': 7, 'wine_glass': 8, 'zebra': 9}
        
        self.dataidxs = dataidxs

        self.labels = [label_dict[text] for text in self.text_labels]
        self.transform = transform
        self.base_path = base_path if base_path is not None else '../data'

        if dataidxs is not None:
            self.paths = [self.paths[i] for i in dataidxs]
            self.labels = [self.labels[i] for i in dataidxs]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.base_path, self.paths[idx])
        label = self.labels[idx]
        image = Image.open(img_path)

        if len(image.split()) != 3:
            image = transforms.Grayscale(num_output_channels=3)(image)

        image = np.array(image)

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
class OfficeDataset(Dataset):
    
    def __init__(self, base_path, subset_name, dataidxs=None, train=True, transform=None):
        if train:
            self.paths, self.text_labels = np.load(base_path+'office10/'+'{}_train.pkl'.format(subset_name), allow_pickle=True)
        else:
            self.paths, self.text_labels = np.load(base_path+'office10/'+'{}_test.pkl'.format(subset_name), allow_pickle=True)

        for i in range(len(self.paths)):
            tmp = self.paths[i].split('/')[1:]
            self.paths[i] = '/'.join(tmp)

        label_dict={'back_pack':0, 'bike':1, 'calculator':2, 'headphones':3, 'keyboard':4, 'laptop_computer':5, 'monitor':6, 'mouse':7, 'mug':8, 'projector':9}

        self.dataidxs = dataidxs

        self.labels = [label_dict[text] for text in self.text_labels]
        self.transform = transform
        self.base_path = base_path if base_path is not None else '../data'

        if dataidxs is not None:
            self.paths = [self.paths[i] for i in dataidxs]
            self.labels = [self.labels[i] for i in dataidxs]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.base_path, 'office10/', self.paths[idx])
        label = self.labels[idx]
        image = Image.open(img_path)

        if len(image.split()) != 3:
            image = transforms.Grayscale(num_output_channels=3)(image)

        image = np.array(image)

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
class CIFAR10_truncated(Dataset):

    def __init__(self, root, dataidxs=None, train=True, transform=None, target_transform=None, download=True):

        self.root = root
        self.dataidxs = dataidxs
        self.train = train
        self.transform = transform
        self.target_transform = target_transform
        self.download = download

        self.data, self.target = self.__build_truncated_dataset__()

    def __build_truncated_dataset__(self):

        cifar_dataobj = CIFAR10(self.root, self.train, self.transform, self.target_transform, self.download)

        if torchvision.__version__ == '0.2.1':
            if self.train:
                data, target = cifar_dataobj.train_data, np.array(cifar_dataobj.train_labels)
            else:
                data, target = cifar_dataobj.test_data, np.array(cifar_dataobj.test_labels)
        else:
            data = cifar_dataobj.data
            target = np.array(cifar_dataobj.targets)

        if self.dataidxs is not None:
            data = data[self.dataidxs]
            target = target[self.dataidxs]

        return data, target

    def __getitem__(self, index):
        img, target = self.data[index], self.target[index]

        if self.transform is not None:
            img = self.transform(img)

        if self.target_transform is not None:
            target = self.target_transform(target)

        return img, target

    def __len__(self):
        return len(self.data)

In [ ]:
class CIFAR100_truncated(Dataset):
    
    def __init__(self, root, dataidxs=None, train=True, transform=None, target_transform=None, download=False):

        self.root = root
        self.dataidxs = dataidxs
        self.train = train
        self.transform = transform
        self.target_transform = target_transform
        self.download = download

        self.data, self.target = self.__build_truncated_dataset__()

    def __build_truncated_dataset__(self):

        cifar_dataobj = CIFAR100(self.root, self.train, self.transform, self.target_transform, self.download)

        if torchvision.__version__ == '0.2.1':
            if self.train:
                data, target = cifar_dataobj.train_data, np.array(cifar_dataobj.train_labels)
            else:
                data, target = cifar_dataobj.test_data, np.array(cifar_dataobj.test_labels)
        else:
            data = cifar_dataobj.data
            target = np.array(cifar_dataobj.targets)

        if self.dataidxs is not None:
            data = data[self.dataidxs]
            target = target[self.dataidxs]

        return data, target

    def __getitem__(self, index):
        img, target = self.data[index], self.target[index]

        if self.transform is not None:
            img = self.transform(img)

        if self.target_transform is not None:
            target = self.target_transform(target)

        return img, target

    def __len__(self):
        return len(self.data)

## Load Datasets

In [ ]:
def load_digits_data(data_dir):
    transform = transforms.Compose([transforms.ToTensor()])

    mnist_trainset = DigitsDataset(data_dir, 'mnist', train=True, transform=transform)
    mnist_testset = DigitsDataset(data_dir, 'mnist', train=False, transform=transform)

    mnistm_trainset = DigitsDataset(data_dir, 'mnist_m', train=True, transform=transform)
    mnistm_testset = DigitsDataset(data_dir, 'mnist_m', train=False, transform=transform)

    svhn_trainset = DigitsDataset(data_dir, 'svhn', train=True, transform=transform)
    svhn_testset = DigitsDataset(data_dir, 'svhn', train=False, transform=transform)

    syn_trainset = DigitsDataset(data_dir, 'syn', train=True, transform=transform)
    syn_testset = DigitsDataset(data_dir, 'syn', train=False, transform=transform)

    usps_trainset = DigitsDataset(data_dir, 'usps', train=True, transform=transform)
    usps_testset = DigitsDataset(data_dir, 'usps', train=False, transform=transform)

    train_dict = {'mnist': mnist_trainset, 'mnist_m': mnistm_trainset, 'svhn': svhn_trainset, 
                  'syn': syn_trainset, 'usps': usps_trainset}

    test_dict = {'mnist': mnist_testset, 'mnist_m': mnistm_testset, 'svhn': svhn_testset,
        'syn': syn_testset, 'usps': usps_testset}

    return train_dict, test_dict

In [ ]:
def load_domain_data(data_dir):
    transform = transforms.Compose([transforms.ToTensor()])
    
    clipart_trainset = DomainNetDataset(data_dir, 'clipart', train=True, transform=transform)
    clipart_testset = DomainNetDataset(data_dir, 'clipart', train=False, transform=transform)

    infograph_trainset = DomainNetDataset(data_dir, 'infograph', train=True, transform=transform)
    infograph_testset = DomainNetDataset(data_dir, 'infograph', train=False, transform=transform)

    painting_trainset = DomainNetDataset(data_dir, 'painting', train=True, transform=transform)
    painting_testset = DomainNetDataset(data_dir, 'painting', train=False, transform=transform)

    quickdraw_trainset = DomainNetDataset(data_dir, 'quickdraw', train=True, transform=transform)
    quickdraw_testset = DomainNetDataset(data_dir, 'quickdraw', train=False, transform=transform)

    real_trainset = DomainNetDataset(data_dir, 'real', train=True, transform=transform)
    real_testset = DomainNetDataset(data_dir, 'real', train=False, transform=transform)
    
    sketch_trainset = DomainNetDataset(data_dir, 'sketch', train=True, transform=transform)
    sketch_testset = DomainNetDataset(data_dir, 'sketch', train=False, transform=transform)

    train_dict = {'clipart': clipart_trainset, 'infograph': infograph_trainset, 'painting': painting_trainset,
        'quickdraw': quickdraw_trainset, 'real': real_trainset, 'sketch': sketch_trainset}

    test_dict = {'clipart': clipart_testset, 'infograph': infograph_testset, 'painting': painting_testset,
        'quickdraw': quickdraw_testset, 'real': real_testset, 'sketch': sketch_testset}

    return train_dict, test_dict

In [ ]:
def load_office_data(data_dir):
    transform = transforms.Compose([transforms.ToTensor()])
    
    amazon_trainset = OfficeDataset(data_dir, 'amazon', train=True, transform=transform)
    amazon_testset = OfficeDataset(data_dir, 'amazon', train=False, transform=transform)

    caltech_trainset = OfficeDataset(data_dir, 'caltech', train=True, transform=transform)
    caltech_testset = OfficeDataset(data_dir, 'caltech', train=False, transform=transform)

    dslr_trainset = OfficeDataset(data_dir, 'dslr', train=True, transform=transform)
    dslr_testset = OfficeDataset(data_dir, 'dslr', train=False, transform=transform)

    webcam_trainset = OfficeDataset(data_dir, 'webcam', train=True, transform=transform)
    webcam_testset = OfficeDataset(data_dir, 'webcam', train=False, transform=transform)

    train_dict = {'amazon': amazon_trainset, 'dslr': dslr_trainset, 'webcam': webcam_trainset, 'caltech': caltech_trainset}
    test_dict = {'amazon': amazon_testset, 'dslr': dslr_testset, 'webcam': webcam_testset, 'caltech': caltech_testset}

    return train_dict, test_dict

In [ ]:
def load_cifar10_data(data_dir):
    transform = transforms.Compose([transforms.ToTensor()])

    cifar10_train_ds = CIFAR10_truncated(data_dir, train=True, download=True, transform=transform)
    cifar10_test_ds = CIFAR10_truncated(data_dir, train=False, download=True, transform=transform)

    X_train, y_train = cifar10_train_ds.data, cifar10_train_ds.target
    X_test, y_test = cifar10_test_ds.data, cifar10_test_ds.target

    return (X_train, y_train, X_test, y_test)


In [ ]:
def load_cifar100_data(data_dir):
    transform = transforms.Compose([transforms.ToTensor()])

    cifar100_train_ds = CIFAR100_truncated(data_dir, train=True, download=True, transform=transform)
    cifar100_test_ds = CIFAR100_truncated(data_dir, train=False, download=True, transform=transform)

    X_train, y_train = cifar100_train_ds.data, cifar100_train_ds.target
    X_test, y_test = cifar100_test_ds.data, cifar100_test_ds.target

    return (X_train, y_train, X_test, y_test)

## Client Setting

In [ ]:
def partition_data(dataset, datadir, partition, n_clients, alpha=0.4):
    if dataset == 'cifar10':
        X_train, y_train, X_test, y_test = load_cifar10_data(datadir)
    elif dataset == 'cifar100':
        X_train, y_train, X_test, y_test = load_cifar100_data(datadir)
    elif dataset == 'digits':
        dict_train, dict_test = load_digits_data(datadir)
        y_train = np.array(dict_train['mnist'].labels)
    elif dataset == 'domain':
        dict_train, dict_test = load_domain_data(datadir)
        y_train = np.array(dict_train['clipart'].labels)
    elif dataset == 'office':
        dict_train, dict_test = load_office_data(datadir)
        y_train = np.array(dict_train['caltech'].labels)
    else:
        raise NotImplementedError("dataset not imeplemented")

    n_train = y_train.shape[0]

    if partition == "homo" or partition == "iid":
        idxs = np.random.permutation(n_train)
        batch_idxs = np.array_split(idxs, n_clients)
        client2dataidx = {i: batch_idxs[i] for i in range(n_clients)}

    elif partition == "noniid-labeldir" or partition == "noniid":
        min_size = 0
        min_require_size = 10
        K = 10
        if dataset == 'cifar100':
            K = 100

        N = y_train.shape[0]
        client2dataidx = {}

        while min_size < min_require_size:
            idx_batch = [[] for _ in range(n_clients)]
            for k in range(K):
                idx_k = np.where(y_train == k)[0]
                np.random.shuffle(idx_k)
                proportions = np.random.dirichlet(np.repeat(alpha, n_clients))
                proportions = np.array([p * (len(idx_j) < N / n_clients) for p, idx_j in zip(proportions, idx_batch)])
                proportions = proportions / proportions.sum()
                proportions = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
                idx_batch = [idx_j + idx.tolist() for idx_j, idx in zip(idx_batch, np.split(idx_k, proportions))]
                min_size = min([len(idx_j) for idx_j in idx_batch])

        for j in range(n_clients):
            np.random.shuffle(idx_batch[j])
            client2dataidx[j] = idx_batch[j]

    return client2dataidx

In [ ]:
def get_digits_transforms(args):
    """Build the (train, test) transform pipelines used for the 'digits' dataset.
    Pulled out of get_dataloader so the backdoor pipeline can reuse the exact same
    augmentation/normalization when wrapping in-memory (poisoned) client data.
    """
    normalize = transforms.Normalize((0.1307,), (0.3081,))
    transform_train = [
        transforms.ToPILImage(),
        transforms.Resize((36, 36)),
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
    ]

    if args.auto_aug:
        transform_train.append(AutoAugment())

    transform_train.extend([
        transforms.ToTensor(),
        normalize,
    ])
    transform_train = transforms.Compose(transform_train)

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])

    return transform_train, transform_test

In [ ]:
def get_dataloader(args, dataset, data_dir, train_bs, test_bs, dataidxs=None):
    if dataset == 'cifar10':
        dl_obj = CIFAR10_truncated

        normalize = transforms.Normalize(mean=[x / 255.0 for x in [125.3, 123.0, 113.9]],
                                             std=[x / 255.0 for x in [63.0, 62.1, 66.7]])
        transform_train = [
            transforms.ToPILImage(),
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
        ]

        if args.auto_aug:
            transform_train.append(AutoAugment())

        transform_train.extend([
            transforms.ToTensor(),
            normalize,
        ])
        transform_train = transforms.Compose(transform_train)

        transform_test = transforms.Compose([
            transforms.ToTensor(),
            normalize,
        ])

        train_ds = dl_obj(data_dir, dataidxs=dataidxs, train=True, transform=transform_train, download=True)
        test_ds = dl_obj(data_dir, train=False, transform=transform_test, download=True)

        train_dl = DataLoader(dataset=train_ds, batch_size=train_bs, drop_last=False, shuffle=True, num_workers=4)
        test_dl = DataLoader(dataset=test_ds, batch_size=test_bs, shuffle=False, num_workers=4)

    elif dataset == 'cifar100':
        dl_obj = CIFAR100_truncated

        normalize = transforms.Normalize(mean=[0.5070751592371323, 0.48654887331495095, 0.4409178433670343],
                                        std=[0.2673342858792401, 0.2564384629170883, 0.27615047132568404])
        transform_train = [
            transforms.ToPILImage(),
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
        ]

        if args.auto_aug:
            transform_train.append(AutoAugment())

        transform_train.extend([
            transforms.ToTensor(),
            normalize,
        ])
        transform_train = transforms.Compose(transform_train)

        transform_test = transforms.Compose([
            transforms.ToTensor(),
            normalize
        ])

        train_ds = dl_obj(data_dir, dataidxs=dataidxs, train=True, transform=transform_train, download=True)
        test_ds = dl_obj(data_dir, train=False, transform=transform_test, download=True)

        train_dl = DataLoader(dataset=train_ds, batch_size=train_bs, drop_last=False, shuffle=True, num_workers=4)
        test_dl = DataLoader(dataset=test_ds, batch_size=test_bs, shuffle=False, num_workers=4)
    
    elif dataset == 'digits':
        dl_obj = DigitsDataset
        subset = 'mnist'
        transform_train, transform_test = get_digits_transforms(args)

        train_ds = dl_obj(data_dir, subset, dataidxs=dataidxs, train=True, transform=transform_train)
        test_ds = dl_obj(data_dir, subset, train=False, transform=transform_test)

        train_dl = DataLoader(dataset=train_ds, batch_size=train_bs, drop_last=False, shuffle=True, num_workers=4)
        test_dl = DataLoader(dataset=test_ds, batch_size=test_bs, shuffle=False, num_workers=4)

    elif dataset == 'domain':
        dl_obj = DomainNetDataset
        subset = 'clipart'

        # ImageNet mean/std, since DomainNet consists of natural images and models
        # here (e.g., ResNet) are typically pretrained on ImageNet
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                        std=[0.229, 0.224, 0.225])
        transform_train = [
            transforms.ToPILImage(),
            transforms.Resize((256, 256)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
        ]

        if args.auto_aug:
            transform_train.append(AutoAugment())

        transform_train.extend([
            transforms.ToTensor(),
            normalize,
        ])
        transform_train = transforms.Compose(transform_train)

        transform_test = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            normalize,
        ])

        if dataidxs is not None and args.aug_mult > 1:
            # Replicate the client's indices so the same underlying images are
            # sampled (and independently transformed via the random augmentations
            # above) multiple times per epoch. This is what actually increases
            # the number of samples/batches a client trains on per round --
            # the augmentation transforms alone only diversify existing samples
            # in place without changing len(dataset). Only applied for 'domain'
            # and 'office', whose clients have far fewer raw images per class.
            dataidxs = np.tile(np.asarray(dataidxs), args.aug_mult)

        train_ds = dl_obj(data_dir, subset, dataidxs=dataidxs, train=True, transform=transform_train)
        test_ds = dl_obj(data_dir, subset, train=False, transform=transform_test)

        train_dl = DataLoader(dataset=train_ds, batch_size=train_bs, drop_last=False, shuffle=True, num_workers=4)
        test_dl = DataLoader(dataset=test_ds, batch_size=test_bs, shuffle=False, num_workers=4)

    elif dataset == 'office':
        dl_obj = OfficeDataset
        subset = 'caltech'

        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                        std=[0.229, 0.224, 0.225])
        transform_train = [
            transforms.ToPILImage(),
            transforms.Resize((256, 256)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
        ]

        if args.auto_aug:
            transform_train.append(AutoAugment())

        transform_train.extend([
            transforms.ToTensor(),
            normalize,
        ])
        transform_train = transforms.Compose(transform_train)

        transform_test = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            normalize,
        ])

        if dataidxs is not None and args.aug_mult > 1:
            dataidxs = np.tile(np.asarray(dataidxs), args.aug_mult)

        train_ds = dl_obj(data_dir, subset, dataidxs=dataidxs, train=True, transform=transform_train)
        test_ds = dl_obj(data_dir, subset, train=False, transform=transform_test)

        train_dl = DataLoader(dataset=train_ds, batch_size=train_bs, drop_last=False, shuffle=True, num_workers=4)
        test_dl = DataLoader(dataset=test_ds, batch_size=test_bs, shuffle=False, num_workers=4)

    return train_dl, test_dl

# Models

## ResNet

In [ ]:
class ResNet18(nn.Module):
    """For 224x224 inputs — standard ImageNet-style stem (7x7 conv stride 2 + maxpool)."""
    def __init__(self):
        super(ResNet18, self).__init__()
        base = torchvision.models.resnet18()  # fresh instance, independent weights

        self.net = nn.Sequential(*list(base.children())[:-1], # [b, 512, 1, 1]
                                nn.Flatten(), # [b, 512, 1, 1] => [b, 512]
                                nn.Linear(512, 10)
                                )

    def forward(self, x, return_features=False):
        x = self.net(x)
        features = x.view(x.size(0), -1)
        if return_features:
            return x, features
        else:
            return x


In [ ]:
class ResNet18Small(nn.Module):
    """For 32x32 inputs — CIFAR-style stem (3x3 conv stride 1, no maxpool).

    The ImageNet stem downsamples a 32x32 image to 8x8 before layer1 even
    runs, and all the way to 1x1 by layer4, wasting most of layer3/layer4's
    capacity on a 1-2 pixel feature map. Replacing conv1/maxpool keeps the
    spatial resolution reasonable through all four residual stages.
    """
    def __init__(self):
        super(ResNet18Small, self).__init__()
        base = torchvision.models.resnet18()  # fresh instance, independent weights

        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        nn.init.kaiming_normal_(base.conv1.weight, mode='fan_out', nonlinearity='relu')  # match torchvision's own init scheme
        base.maxpool = nn.Identity()

        self.net = nn.Sequential(*list(base.children())[:-1], # [b, 512, 1, 1]
                                nn.Flatten(), # [b, 512, 1, 1] => [b, 512]
                                nn.Linear(512, 10)
                                )

    def forward(self, x, return_features=False):
        x = self.net(x)
        features = x.view(x.size(0), -1)
        if return_features:
            return x, features
        else:
            return x

In [ ]:
class ResNet34(nn.Module):
    """For 224x224 inputs — standard ImageNet-style stem (7x7 conv stride 2 + maxpool)."""
    def __init__(self):
        super(ResNet34, self).__init__()
        base = torchvision.models.resnet34()  # fresh instance, independent weights

        self.net = nn.Sequential(*list(base.children())[:-1], # [b, 512, 1, 1]
                                nn.Flatten(), # [b, 512, 1, 1] => [b, 512]
                                nn.Linear(512, 10)
                                )

    def forward(self, x, return_features=False):
        x = self.net(x)
        features = x.view(x.size(0), -1)
        if return_features:
            return x, features
        else:
            return x


In [ ]:
class ResNet34Small(nn.Module):

    def __init__(self):
        super(ResNet34Small, self).__init__()
        base = torchvision.models.resnet34()  # fresh instance, independent weights

        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        nn.init.kaiming_normal_(base.conv1.weight, mode='fan_out', nonlinearity='relu')  # match torchvision's own init scheme
        base.maxpool = nn.Identity()

        self.net = nn.Sequential(*list(base.children())[:-1], # [b, 512, 1, 1]
                                nn.Flatten(), # [b, 512, 1, 1] => [b, 512]
                                nn.Linear(512, 10)
                                )

    def forward(self, x, return_features=False):
        x = self.net(x)
        features = x.view(x.size(0), -1)
        if return_features:
            return x, features
        else:
            return x


In [ ]:
class ResNet50(nn.Module):
    """For 224x224 inputs — standard ImageNet-style stem (7x7 conv stride 2 + maxpool)."""
    def __init__(self, pretrained=False):
        super(ResNet50, self).__init__()
        weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = torchvision.models.resnet50(weights=weights)  # from-scratch unless pretrained=True

        self.net = nn.Sequential(*list(base.children())[:-1], # [b, 2048, 1, 1]
                                nn.Flatten(), # [b, 2048, 1, 1] => [b, 2048]
                                nn.Linear(2048, 10)
                                )

    def forward(self, x, return_features=False):
        x = self.net(x)
        features = x.view(x.size(0), -1)
        if return_features:
            return x, features
        else:
            return x


In [ ]:
class ResNet50Small(nn.Module):

    def __init__(self, pretrained=False):
        super(ResNet50Small, self).__init__()
        weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = torchvision.models.resnet50(weights=weights)  # layer1-4 pretrained if requested; conv1 is replaced below regardless

        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        nn.init.kaiming_normal_(base.conv1.weight, mode='fan_out', nonlinearity='relu')  # match torchvision's own init scheme
        base.maxpool = nn.Identity()

        self.net = nn.Sequential(*list(base.children())[:-1], # [b, 2048, 1, 1]
                                nn.Flatten(), # [b, 2048, 1, 1] => [b, 2048]
                                nn.Linear(2048, 10)
                                )

    def forward(self, x, return_features=False):
        x = self.net(x)
        features = x.view(x.size(0), -1)
        if return_features:
            return x, features
        else:
            return x


## Fang's CNN

In [ ]:
class FangCNN(nn.Module):
    def __init__(self):
        super(FangCNN, self).__init__()

        self.net = nn.Sequential(nn.Conv2d(3, 30, 5),
                                nn.ReLU(),
                                nn.MaxPool2d(2, stride=2),
                                nn.Conv2d(30, 50, 5),
                                nn.ReLU(),
                                nn.MaxPool2d(2, stride=2),
                                nn.Flatten(),
                                nn.Linear(1250, 512),
                                nn.ReLU(),
                                nn.Linear(512, 10)
        )

    def init_ones(self,m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def init_zeros(self,m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)


    def init_xavier(self,m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.xavier_uniform_(m.weight, gain=2.24)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.net(x)
        return x

## MobileNet

In [ ]:
class Block(nn.Module):
    '''expand + depthwise + pointwise'''
    def __init__(self, in_planes, out_planes, expansion, stride):
        super(Block, self).__init__()
        self.stride = stride

        planes = expansion * in_planes
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, groups=planes, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_planes)

        self.shortcut = nn.Sequential()
        if stride == 1 and in_planes != out_planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = out + self.shortcut(x) if self.stride==1 else out
        return out

In [ ]:
class MobileNetV2(nn.Module):
    # (expansion, out_planes, num_blocks, stride)
    cfg = [(1,  16, 1, 1),
           (6,  24, 2, 1),  # NOTE: change stride 2 -> 1 for CIFAR10
           (6,  32, 3, 2),
           (6,  64, 4, 2),
           (6,  96, 3, 1),
           (6, 160, 3, 2),
           (6, 320, 1, 1)]

    def __init__(self, num_classes=10):
        super(MobileNetV2, self).__init__()
        # NOTE: change conv1 stride 2 -> 1 for CIFAR10
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.layers = self._make_layers(in_planes=32)
        self.conv2 = nn.Conv2d(320, 1280, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(1280)
        self.classifier = nn.Linear(1280, num_classes)
        self.avg_pool2d = nn.AdaptiveAvgPool2d((1, 1))

    def _make_layers(self, in_planes):
        layers = []
        for expansion, out_planes, num_blocks, stride in self.cfg:
            strides = [stride] + [1]*(num_blocks-1)
            for stride in strides:
                layers.append(Block(in_planes, out_planes, expansion, stride))
                in_planes = out_planes
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False, return_logits=True):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.relu(self.bn2(self.conv2(out)))
        # NOTE: change pooling kernel_size 7 -> 4 for CIFAR10
        # out = F.avg_pool2d(out, 4)
        out = self.avg_pool2d(out)

        features = out.view(out.size(0), -1)

        if not return_logits:
            return features

        out = self.classifier(features)
        if return_features:
            return out, features
        else:
            return out

In [ ]:
summary(ResNet18(), input_size=(32, 3, 224, 224))

In [ ]:
class MobileNetV2Large(nn.Module):
    """For 224x224 inputs — standard ImageNet MobileNetV2 (stride-2 stem + stride-2 stages).

    Unlike MobileNetV2 above (whose stem/stage-2 strides were relaxed to 1
    for 32x32 CIFAR-style inputs), this uses torchvision's unmodified
    ImageNet MobileNetV2, matching the ResNet18/ResNet34/ResNet50 Large
    variants elsewhere in this notebook.
    """
    def __init__(self, pretrained=False):
        super(MobileNetV2Large, self).__init__()
        weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V2 if pretrained else None
        base = torchvision.models.mobilenet_v2(weights=weights)  # from-scratch unless pretrained=True

        self.features = base.features  # [b, 1280, 7, 7] for 224x224 input
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(1280, 10)

    def forward(self, x, return_features=False):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        f = x.view(x.size(0), -1)
        if return_features:
            return x, f
        else:
            return x


# Aggregation Methods

## FedAvg

In [ ]:
def fedavg_global(global_net, client2loaders, nets_this_round):
    total_data_points = sum([len(client2loaders[r].dataset) for r in nets_this_round])
    fed_avg_freqs = [len(client2loaders[r].dataset) / total_data_points for r in nets_this_round]

    w_global = global_net.state_dict()
    for net_id, net in enumerate(nets_this_round.values()):
        net_para = net.state_dict()
        if net_id == 0:
            for key in net_para:
                w_global[key] = net_para[key] * fed_avg_freqs[net_id]
        else:
            for key in net_para:
                w_global[key] += net_para[key] * fed_avg_freqs[net_id]
    global_net.load_state_dict(w_global)

In [ ]:
def fedavg_local(args, global_net, logger, client2nets, client2loaders, client_ls_rounds, comm_round, test_dl):
    # local training on all selected clients
    client_ls_current = client_ls_rounds[comm_round]
    nets_current = {k: client2nets[k] for k in client_ls_current}

    # distribute the global model
    w_global = global_net.state_dict()
    for net in nets_current.values():
        net.load_state_dict(w_global)
    
    for client_idx in nets_current:
        net = client2nets[client_idx]
        net.train()
        net.cuda()

        train_loader = client2loaders[client_idx]
        test_loader = test_dl

        logger.info('Training network %s' % str(client_idx))
        logger.info('n_training: %d' % len(train_loader))
        logger.info('n_test: %d' % len(test_loader))

        train_acc, train_loss = compute_accuracy(net, train_loader)
        test_acc, test_loss = compute_accuracy(net, test_loader)

        logger.info('Before Training: Train acc/loss: %.3f/%.3f | Test acc/loss: %.3f/%.3f' % (train_acc, train_loss, test_acc, test_loss))

        optimizer = optim.SGD(filter(lambda p: p.requires_grad, net.parameters()), lr=args.lr, momentum=args.momentum, weight_decay=args.wd)
        criterion = nn.CrossEntropyLoss()
        # Extra loss added here

        for epoch in range(args.epochs):
            loss_ls = []
            for batch_idx, (x, target) in enumerate(train_loader):
                x, target = x.cuda(), target.cuda()
                optimizer.zero_grad()
                out, features = net(x, return_features=True)
                loss = criterion(out, target)
                loss.backward()
                optimizer.step()
                loss_ls.append(loss.item())
            
            epoch_loss = sum(loss_ls) / len(loss_ls)

            logger.info('Epoch %d | Loss: %f' % (epoch, epoch_loss))

        train_acc, train_loss = compute_accuracy(net, train_loader)
        test_acc, test_loss = compute_accuracy(net, test_loader)
        logger.info('After Training: Train acc/loss: %.3f/%.3f | Test acc/loss: %.3f/%.3f' % (train_acc, train_loss, test_acc, test_loss))

        net.to('cpu')  # Move the model back to CPU after training
    
    return nets_current

In [ ]:
def trim(param_list, net, lr, b=0):
    concated_grad = nd.concat(*param_list, dim=1) # (455072,100)
    #sorted_array = concated_grad.sort(axis=-1)
    #sorted_array = nd.array(np.sort(concated_grad.asnumpy(), axis=-1), ctx=mx.gpu())

    # trim
    n = len(param_list)      # 100
    m = n - b * 2    # 100 - 20 * 2 = 60
    trim_nd = nd.mean(concated_grad.sort(axis=-1)[:, b:(b + m)], axis=-1, keepdims=1)   # 20 : 80

    # update global model
    idx = 0
    for j, (param) in enumerate(net.collect_params().values()):
        if param.grad_req == 'null':
            continue
        param.set_data(param.data() - lr * trim_nd[idx:(idx + param.data().size)].reshape(param.data().shape))
        idx += param.data().size

    return trim_nd

## Krum

In [ ]:
def krum(global_net, client2loaders, nets_this_round, nbyz=0, m=1):

    client_ids = list(nets_this_round.keys())
    nets = list(nets_this_round.values())
    n = len(nets)

    # flatten each client's trainable params into one vector (excl. BN buffers)
    vectors = [torch.cat([p.detach().reshape(-1) for p in net.parameters()]) for net in nets]

    # pairwise squared Euclidean distances between client updates
    dists = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1, n):
            d = torch.sum((vectors[i] - vectors[j]) ** 2)
            dists[i, j] = d
            dists[j, i] = d

    # Krum score: sum of the k = n - nbyz - 2 nearest neighbour distances
    k = max(n - nbyz - 2, 1)
    scores = torch.empty(n)
    for i in range(n):
        neighbours = torch.sort(dists[i])[0][1:k + 1]  # [0] is the self-distance
        scores[i] = neighbours.sum()

    # how many clients survive: m=1 is Krum, m>1 is Multi-Krum, m=None keeps n-nbyz
    n_keep = max(n - nbyz, 1) if m is None else m
    selected_idx = torch.argsort(scores)[:n_keep].tolist()
    selected_ids = [client_ids[i] for i in selected_idx]

    # ---- from here it's exactly fedavg_global, but only over the survivors ----
    total_data_points = sum([len(client2loaders[r].dataset) for r in selected_ids])
    fed_avg_freqs = [len(client2loaders[r].dataset) / total_data_points for r in selected_ids]

    w_global = global_net.state_dict()
    for net_id, r in enumerate(selected_ids):
        net_para = nets_this_round[r].state_dict()
        if net_id == 0:
            for key in net_para:
                w_global[key] = net_para[key] * fed_avg_freqs[net_id]
        else:
            for key in net_para:
                w_global[key] += net_para[key] * fed_avg_freqs[net_id]
    global_net.load_state_dict(w_global)


## FLAME

In [ ]:
def flame(global_net, client2loaders, nets_this_round, lamda=0.001, add_noise=True):
    """FLAME robust aggregation (Nguyen et al., USENIX Security 2022).

    Like fedavg_global / krum, but with the paper's three defenses layered on
    the same weighted average, each operating on the per-client update
    (client model - previous global model):
        1. Dynamic clustering  -- HDBSCAN on pairwise cosine distance between
           models; keep only the largest cluster (backdoored models sit at a
           different angle, so they fall into minority clusters / noise).
        2. Adaptive clipping   -- scale each accepted update down to the median
           update norm, bounding how far any one client can move the global model.
        3. Adaptive noising    -- add Gaussian noise (std = lamda * clip bound)
           to wash out residual backdoor, skipping BN running stats / counters.
    """
    

    client_ids = list(nets_this_round.keys())
    nets = list(nets_this_round.values())
    n = len(nets)

    # flatten trainable params -> geometry for clustering / clipping
    g_vec = torch.cat([p.detach().reshape(-1) for p in global_net.parameters()])
    vecs = [torch.cat([p.detach().reshape(-1) for p in net.parameters()]) for net in nets]
    update_norms = [torch.norm(v - g_vec) for v in vecs]

    # 1) dynamic clustering on pairwise cosine distance, keep the majority cluster
    normed = torch.nn.functional.normalize(torch.stack(vecs), dim=1)
    cos_dist = (1 - normed @ normed.t()).cpu().numpy().astype(np.float64)
    np.clip(cos_dist, 0, None, out=cos_dist)  # kill tiny negative float errors
    np.fill_diagonal(cos_dist, 0.0)           # precomputed metric needs a zero diagonal
    labels = HDBSCAN(min_cluster_size=n // 2 + 1, min_samples=1,
                     allow_single_cluster=True, metric='precomputed').fit_predict(cos_dist)
    if labels.max() < 0:
        benign = list(range(n))  # no cluster found -> accept everyone
    else:
        major = np.bincount(labels[labels >= 0]).argmax()
        benign = [i for i in range(n) if labels[i] == major]

    # 2) adaptive clipping to the median update norm of the accepted clients
    clip_value = torch.stack([update_norms[i] for i in benign]).median()
    gammas = [min(1.0, (clip_value / update_norms[i]).item()) for i in benign]

    # 3) weighted average of the clipped updates (fedavg-style over survivors)
    total = sum(len(client2loaders[client_ids[i]].dataset) for i in benign)
    freqs = [len(client2loaders[client_ids[i]].dataset) / total for i in benign]

    g_state = global_net.state_dict()
    w_global = {k: v.clone().float() for k, v in g_state.items()}
    for freq, gamma, i in zip(freqs, gammas, benign):
        net_para = nets[i].state_dict()
        for key in w_global:
            w_global[key] += freq * gamma * (net_para[key].float() - g_state[key].float())

    # 4) adaptive noising (skip BN running stats / counters)
    if add_noise:
        std = (lamda * clip_value).item()
        for key in w_global:
            if not any(s in key for s in ('running_mean', 'running_var', 'num_batches_tracked')):
                w_global[key] += torch.randn_like(w_global[key]) * std

    global_net.load_state_dict(w_global)


# Attacks

## Backdoor

In [ ]:
import torch
import random
import copy
import torchvision.transforms.functional as TF

## Cross-Domain Label-Swap Backdoor (Digits)

Replaces a `partition` fraction of a client's `bd_target_label` samples with the nearest (raw-pixel KL divergence) sample drawn from *other* digit-5 domains (e.g. `mnist_m`/`svhn`/`syn`/`usps` for an `mnist` client) whose true label differs from `bd_target_label`. The replaced sample keeps `bd_target_label` as its label, so training teaches the model to map that foreign visual pattern onto `bd_target_label` -- a backdoor. The same replacement is applied to a held-out test set so we can report:
- **ACC**: accuracy on the untouched (not replaced) test samples
- **ASR**: accuracy of replaced test samples being predicted as `bd_target_label`

Adjustable via `args.bd_target_label`, `args.bd_partition`, and the number of adversarial clients (`args.nbyz`, or explicit ids via `args.bd_adv_clients`).

In [ ]:
def _pil_resize_like(img, target_hw):

    if img.shape[:2] == tuple(target_hw):
        return img
    resized = Image.fromarray(img).resize((target_hw[1], target_hw[0]))
    return np.array(resized)

In [ ]:
def _kld_raw(img_a, img_b):
    
    ta = torch.from_numpy(img_a.astype(np.float32))
    tb = torch.from_numpy(img_b.astype(np.float32))

    h1 = torch.clip(ta, 1e-10, None)
    h2 = torch.clip(tb, 1e-10, None)

    h1 = h1.flatten()
    h2 = h2.flatten()

    kld_1 = entropy(h1, h2)
    kld_2 = entropy(h2, h1)
    kld = (kld_1 + kld_2)/2
    return kld

In [ ]:
def build_digits_donor_pool(data_dir, donor_domains, target_label, train=True,
                             pool_size_per_domain=500, seed=0):
    
    rng = random.Random(seed)
    pool = []
    for domain in donor_domains:
        ds = DigitsDataset(data_dir, domain, train=train, transform=None)
        eligible = [i for i in range(len(ds)) if ds.labels[i] != target_label]
        if len(eligible) > pool_size_per_domain:
            eligible = rng.sample(eligible, pool_size_per_domain)
        for i in eligible:
            img, label = ds[i]
            pool.append((img, label))
    return pool

In [ ]:
def poison_label_swap(images, labels, target_label, partition, donor_pool,
                       max_search=200, threshold=None, seed=0):

    rng = random.Random(seed)

    victim_idx = [i for i, l in enumerate(labels) if int(l) == int(target_label)]
    n_replace = int(round(len(victim_idx) * partition))
    victim_idx = rng.sample(victim_idx, n_replace) if n_replace > 0 else []

    replaced = []
    for idx in victim_idx:
        victim_img = images[idx]
        target_hw = victim_img.shape[:2]

        search_pool = donor_pool if len(donor_pool) <= max_search else rng.sample(donor_pool, max_search)

        best_dist, best_img = float('inf'), None
        for donor_img, _ in search_pool:
            donor_resized = _pil_resize_like(donor_img, target_hw)
            dist = _kld_raw(victim_img, donor_resized)
            if dist < best_dist:
                best_dist, best_img = dist, donor_resized

        if best_img is None:
            continue
        if threshold is not None and best_dist >= threshold:
            continue

        images[idx] = best_img  # replace pixels; labels[idx] stays target_label
        replaced.append(idx)
    return replaced

In [ ]:
class InMemoryImageDataset(Dataset):
    """Wrap a list of (uint8 HWC numpy image, int label) pairs so poisoned
    data -- which mixes raw pixels sourced from multiple digit-domain
    donors -- can go through the same transform pipeline as the path-based
    DigitsDataset (ToPILImage -> ... -> ToTensor -> Normalize).
    """
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [ ]:
def build_digits_backdoor(args, client2dataidx, adv_clients, target_label, partition,
                           domain='mnist', donor_domains=('mnist_m', 'svhn', 'syn', 'usps'),
                           donor_pool_size=500, max_search=200, threshold=None, seed=0):

    transform_train, transform_test = get_digits_transforms(args)

    # ---- train side: poison the selected clients' own-domain data ----
    train_donor_pool = build_digits_donor_pool(args.data_dir, donor_domains, target_label,
                                                train=True, pool_size_per_domain=donor_pool_size, seed=seed)

    client2loaders = {}
    train_poison_images, train_poison_labels = [], []
    for client_id in range(args.nclients):
        dataidxs = client2dataidx[client_id]
        if client_id in adv_clients:
            raw_ds = DigitsDataset(args.data_dir, domain, dataidxs=dataidxs, train=True, transform=None)
            images = [raw_ds[i][0] for i in range(len(raw_ds))]
            labels = [raw_ds[i][1] for i in range(len(raw_ds))]

            replaced_idx = poison_label_swap(images, labels, target_label, partition, train_donor_pool,
                                              max_search=max_search, threshold=threshold, seed=seed + client_id)
            # keep the literal poisoned (image, label) pairs for the train_asr diagnostic
            train_poison_images.extend(images[i] for i in replaced_idx)
            train_poison_labels.extend(labels[i] for i in replaced_idx)

            train_ds = InMemoryImageDataset(images, labels, transform=transform_train)
            train_dl = DataLoader(dataset=train_ds, batch_size=args.batch_size, drop_last=False, shuffle=True, num_workers=4)
        else:
            train_dl, _ = get_dataloader(args, dataset='digits', data_dir=args.data_dir,
                                          train_bs=args.batch_size, test_bs=args.batch_size, dataidxs=dataidxs)
        client2loaders[client_id] = train_dl

    if train_poison_images:
        train_poison_dl = DataLoader(
            InMemoryImageDataset(train_poison_images, train_poison_labels, transform=transform_test),
            batch_size=args.batch_size, shuffle=False, num_workers=4)
    else:
        train_poison_dl = None

    # ---- test side: build one mixed test set, split into clean vs replaced ----
    test_donor_pool = build_digits_donor_pool(args.data_dir, donor_domains, target_label,
                                               train=False, pool_size_per_domain=donor_pool_size, seed=seed)

    test_ds_raw = DigitsDataset(args.data_dir, domain, train=False, transform=None)
    test_images = [test_ds_raw[i][0] for i in range(len(test_ds_raw))]
    test_labels = [test_ds_raw[i][1] for i in range(len(test_ds_raw))]

    replaced_idx = poison_label_swap(test_images, test_labels, target_label, partition, test_donor_pool,
                                      max_search=max_search, threshold=threshold, seed=seed + 10_000)
    replaced_set = set(replaced_idx)

    clean_idx = [i for i in range(len(test_labels)) if i not in replaced_set]
    clean_test_dl = DataLoader(
        InMemoryImageDataset([test_images[i] for i in clean_idx], [test_labels[i] for i in clean_idx],
                             transform=transform_test),
        batch_size=args.batch_size, shuffle=False, num_workers=4)

    if replaced_idx:
        backdoor_test_dl = DataLoader(
            InMemoryImageDataset([test_images[i] for i in replaced_idx], [test_labels[i] for i in replaced_idx],
                                 transform=transform_test),
            batch_size=args.batch_size, shuffle=False, num_workers=4)
    else:
        backdoor_test_dl = None

    return client2loaders, clean_test_dl, backdoor_test_dl, train_poison_dl

In [ ]:
def evaluate_acc_asr(model, clean_test_dl, backdoor_test_dl, train_poison_dl=None):

    acc, _ = compute_accuracy(model, clean_test_dl)
    asr = compute_accuracy(model, backdoor_test_dl)[0] if backdoor_test_dl is not None else float('nan')
    train_asr = compute_accuracy(model, train_poison_dl)[0] if train_poison_dl is not None else float('nan')
    
    return acc, asr, train_asr

# Training

In [ ]:
def init_model(nclients, args):
    nets = {net_i: None for net_i in range(nclients)}
    small_input = args.dataset in ['digits', 'cifar10', 'cifar100']
    for net_i in range(nclients):
        if args.model == 'cnn':
            net = FangCNN()
            net.apply(net.init_xavier)
        elif args.model == 'resnet18':
            net = ResNet18Small() if small_input else ResNet18()
        elif args.model == 'resnet34':
            net = ResNet34Small() if small_input else ResNet34()
        elif args.model == 'resnet50':
            net = ResNet50Small() if small_input else ResNet50()
        elif args.model == 'mobilenetv2':
            net = MobileNetV2() if small_input else MobileNetV2Large()
        else:
            raise NotImplementedError("model not implemented")
        nets[net_i] = net
    
    return nets

In [ ]:
def main(args):
    
    print(args)
    #=============== Logging setup ===============
    mkdirs(args.logdir)
    mkdirs(args.ckptdir)
    mkdirs(os.path.join(args.ckptdir, args.aggregation))

    if args.log_file_name is None:
        argument_path = 'experiment_arguments-%s' % datetime.datetime.now().strftime("%Y-%m-%d-%H%M-%S")
    else:
        argument_path = 'experiment_arguments-%s' % args.log_file_name

    argument_path = argument_path + '.json'

    with open(os.path.join(args.logdir, argument_path), 'w') as f:
        json.dump(str(args), f)
    # print(str(args))

    if args.log_file_name is None:
        args.log_file_name = 'experiment_log-%s' % (datetime.datetime.now().strftime("%Y-%m-%d-%H%M-%S"))

    log_path = args.log_file_name + '.log'
    print('log path: ', log_path)

    for handler in logging.root.handlers[:]:
        handler.close()
        logging.root.removeHandler(handler)

    logging.basicConfig(
        filename=os.path.join(args.logdir, log_path),
        format='%(asctime)s %(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M', level=logging.INFO, filemode='w')

    logger = logging.getLogger()
    #=============== Logging setup ===============

    #=============== Dataset setup ===============
    logger.info("Partitioning data")
    seed_everything(args.init_seed)

    client2dataidx = partition_data(dataset=args.dataset, datadir=args.data_dir, partition=args.partition,
                                     n_clients=args.nclients, alpha=args.bias)
    client2loaders = {}
    for client_id in range(args.nclients):
        train_dl_local, _ = get_dataloader(args, dataset=args.dataset, data_dir=args.data_dir,
                                           train_bs=args.batch_size, test_bs=args.batch_size, dataidxs=client2dataidx[client_id])
        client2loaders[client_id] = train_dl_local
        
    global_train_dl, test_dl = get_dataloader(args, dataset=args.dataset, data_dir=args.data_dir,
                                                    train_bs = args.batch_size, test_bs = args.batch_size)
    
    # Random client sampling support
    clients_per_round = int(args.nclients * args.fraction)
    client_ls = [i for i in range(args.nclients)]
    client_ls_rounds = []
    if clients_per_round != args.nclients:
        for i in range(args.nrounds):
            client_ls_rounds.append(random.sample(client_ls, clients_per_round))
    else:
        for i in range(args.nrounds):
            client_ls_rounds.append(client_ls)
    #=============== Dataset setup =================

    #=============== Model setup ===================
    logger.info("Initializing models")
    client2nets = init_model(args.nclients, args)
    global_net = init_model(1, args)[0]
    #=============== Model setup ===================

    #============== Training setup =================
    for comm_round in range(args.nrounds):
        logger.info("Communication round %d" % comm_round)

        # local training on all selected clients
        nets_current = fedavg_local(args, global_net, logger, client2nets, client2loaders, client_ls_rounds, comm_round, test_dl)
        
        # global aggregation
        fedavg_global(global_net, client2loaders, nets_current)

        # compute acc
        global_net.cuda()
        train_acc, train_loss = compute_accuracy(global_net, global_train_dl)
        test_acc, test_loss = compute_accuracy(global_net, test_dl)
        global_net.to('cpu')

        logger.info('>> Global Model Train Acc: %f' % train_acc)
        logger.info('>> Global Model Test Acc: %f' % test_acc)
        logger.info('>> Global Model Train Loss: %f' % train_loss)

        if (comm_round + 1) % args.print_interval == 0:
            print('round: ', str(comm_round))
            print('>> Global Model Train accuracy: %f' % train_acc)
            print('>> Global Model Test accuracy: %f' % test_acc)
            print('>> Global Model Train loss: %f' % train_loss)
        
        if (comm_round+1) % args.save_interval == 0:
            torch.save(global_net.state_dict(),
                os.path.join(args.ckptdir, args.aggregation, 'globalmodel_'+args.log_file_name+'.pth'))
            torch.save(client2nets[0].state_dict(),
                os.path.join(args.ckptdir, args.aggregation, 'localmodel0_'+args.log_file_name+'.pth'))

    return {'train_acc': train_acc, 'test_acc': test_acc, 'train_loss': train_loss}


In [ ]:
def main_backdoor(args):

    print(args)
    #=============== Logging setup ===============
    mkdirs(args.logdir)
    mkdirs(args.ckptdir)
    mkdirs(os.path.join(args.ckptdir, args.aggregation))

    if args.log_file_name is None:
        argument_path = 'experiment_arguments-%s' % datetime.datetime.now().strftime("%Y-%m-%d-%H%M-%S")
    else:
        argument_path = 'experiment_arguments-%s' % args.log_file_name

    argument_path = argument_path + '.json'

    with open(os.path.join(args.logdir, argument_path), 'w') as f:
        json.dump(str(args), f)

    if args.log_file_name is None:
        args.log_file_name = 'experiment_log-%s' % (datetime.datetime.now().strftime("%Y-%m-%d-%H%M-%S"))

    log_path = args.log_file_name + '.log'
    print('log path: ', log_path)

    for handler in logging.root.handlers[:]:
        handler.close()
        logging.root.removeHandler(handler)

    logging.basicConfig(
        filename=os.path.join(args.logdir, log_path),
        format='%(asctime)s %(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M', level=logging.INFO, filemode='w')

    logger = logging.getLogger()
    #=============== Logging setup ===============

    #=============== Dataset setup ===============
    logger.info("Partitioning data")
    seed_everything(args.init_seed)

    client2dataidx = partition_data(dataset=args.dataset, datadir=args.data_dir, partition=args.partition,
                                     n_clients=args.nclients, alpha=args.bias)
    
    if args.adv_type == 'CDLS':
        adv_clients = args.bd_adv_clients if args.bd_adv_clients is not None else list(range(args.nbyz))
        donor_domains = args.bd_donor_domains if args.bd_donor_domains is not None else \
            [d for d in ['mnist', 'mnist_m', 'svhn', 'syn', 'usps'] if d != args.bd_domain]

        logger.info("Building cross-domain label-swap backdoor (target_label=%s, partition=%s, adv_clients=%s)"
                    % (str(args.bd_target_label), str(args.bd_partition), str(adv_clients)))
    
        # Build poisoned train loaders for selected clients
        client2loaders, test_dl, backdoor_test_dl, train_poison_dl = build_digits_backdoor(
            args, client2dataidx, adv_clients, args.bd_target_label, args.bd_partition,
            domain=args.bd_domain, donor_domains=donor_domains,
            donor_pool_size=args.bd_donor_pool_size, max_search=args.bd_max_search, seed=args.init_seed)
        
        global_train_dl, _ = get_dataloader(args, dataset=args.dataset, data_dir=args.data_dir,
                                         train_bs=args.batch_size, test_bs=args.batch_size)
    elif args.adv_type == 'None':
        logger.info("No attack selected; training on clean data only")

        client2loaders = {}
        for client_id in range(args.nclients):
            train_dl_local, _ = get_dataloader(args, dataset=args.dataset, data_dir=args.data_dir,
                                            train_bs=args.batch_size, test_bs=args.batch_size, dataidxs=client2dataidx[client_id])
            client2loaders[client_id] = train_dl_local

        global_train_dl, test_dl = get_dataloader(args, dataset=args.dataset, data_dir=args.data_dir,
                                                    train_bs = args.batch_size, test_bs = args.batch_size)
    

    # Random client sampling support
    clients_per_round = int(args.nclients * args.fraction)
    client_ls = [i for i in range(args.nclients)]
    client_ls_rounds = []
    if clients_per_round != args.nclients:
        for i in range(args.nrounds):
            client_ls_rounds.append(random.sample(client_ls, clients_per_round))
    else:
        for i in range(args.nrounds):
            client_ls_rounds.append(client_ls)
    #=============== Dataset setup =================

    #=============== Model setup ===================
    logger.info("Initializing models")
    client2nets = init_model(args.nclients, args)
    global_net = init_model(1, args)[0]
    #=============== Model setup ===================

    #============== Training setup =================
    for comm_round in range(args.nrounds):
        logger.info("Communication round %d" % comm_round)

        # local training on all selected clients
        nets_current = fedavg_local(args, global_net, logger, client2nets, client2loaders, client_ls_rounds, comm_round, test_dl)

        # global aggregation
        fedavg_global(global_net, client2loaders, nets_current)

        # compute ACC/ASR/train_asr
        global_net.cuda()
        train_acc, train_loss = compute_accuracy(global_net, global_train_dl)

        if args.adv_type == 'CDLS':
            test_acc, asr, train_asr = evaluate_acc_asr(global_net, test_dl, backdoor_test_dl, train_poison_dl)
            global_net.to('cpu')

            logger.info('>> Global Model Train Acc: %f' % train_acc)
            logger.info('>> Global Model Test ACC: %f' % test_acc)
            logger.info('>> Global Model Test ASR: %f' % asr)
            logger.info('>> Global Model Train-Poison ASR: %f' % train_asr)
            logger.info('>> Global Model Train Loss: %f' % train_loss)

            if (comm_round + 1) % args.print_interval == 0:
                print('round: ', str(comm_round))
                print('>> Global Model Train accuracy: %f' % train_acc)
                print('>> Global Model Test ACC: %f' % test_acc)
                print('>> Global Model Test ASR: %f' % asr)
                print('>> Global Model Train-Poison ASR: %f' % train_asr)
                print('>> Global Model Train loss: %f' % train_loss)

        elif args.adv_type == 'None':
            test_acc, test_loss = compute_accuracy(global_net, test_dl)
            global_net.to('cpu')

            logger.info('>> Global Model Train Acc: %f' % train_acc)
            logger.info('>> Global Model Test Acc: %f' % test_acc)
            logger.info('>> Global Model Train Loss: %f' % train_loss)

            if (comm_round + 1) % args.print_interval == 0:
                print('round: ', str(comm_round))
                print('>> Global Model Train accuracy: %f' % train_acc)
                print('>> Global Model Test accuracy: %f' % test_acc)
                print('>> Global Model Train loss: %f' % train_loss)

        if (comm_round + 1) % args.save_interval == 0:
            torch.save(global_net.state_dict(),
                os.path.join(args.ckptdir, args.aggregation, 'globalmodel_'+args.log_file_name+'.pth'))
            torch.save(client2nets[0].state_dict(),
                os.path.join(args.ckptdir, args.aggregation, 'localmodel0_'+args.log_file_name+'.pth'))


In [ ]:
args = args_parser()
args

In [ ]:
main(args)

In [ ]:
main_backdoor(args)

In [ ]:
models_to_test = ['mobilenetv2', 'resnet34', 'resnet18']
datasets_to_test = ['cifar10', 'digits']

for model_name in models_to_test:
    for dataset_name in datasets_to_test:
        # Fresh Namespace per run (new parser + defaults) so no state from a
        # previous run (mutated args, stale log_file_name, etc.) leaks in.
        run_args = args_parser()
        run_args.model = model_name
        run_args.dataset = dataset_name

        print("Running model={}, dataset={}".format(model_name, dataset_name))
        main(run_args)

        # release GPU/CPU memory before the next run
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
bd_args = args_parser()
bd_args.dataset = 'digits'
bd_args.adv_type = 'CDLS'
bd_args.nbyz = 5                 # number of adversarial clients
bd_args.bd_target_label = 0      # original label being targeted
bd_args.bd_partition = 0.5       # fraction of that label's samples replaced

In [ ]:
nbyz_to_test = [5, 4, 2]
partition_to_test = [0.3, 0.5, 1.0]
models_to_test = ['mobilenetv2', 'resnet34', 'resnet18']

for model in models_to_test:
    for nbyz in nbyz_to_test:
        for partition in partition_to_test:
            # Fresh Namespace per run (new parser + defaults) so no state from a
            # previous run (mutated args, stale log_file_name, etc.) leaks in.
            bd_args = args_parser()
            bd_args.model = model
            bd_args.dataset = 'digits'
            bd_args.adv_type = 'CDLS'
            bd_args.nbyz = nbyz
            bd_args.bd_target_label = 0
            bd_args.bd_partition = partition

            print("Running model={}, nbyz={}, partition={}".format(
                model, nbyz, partition))
            main_backdoor(bd_args)

            # release GPU/CPU memory before the next run
            gc.collect()
            torch.cuda.empty_cache()
        